In [1]:
pip install requests beautifulsoup4 pandas numpy matplotlib seaborn prophet jupyter

   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
    --------------------------------------- 0.3/12.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.1 MB 1.3 MB/s eta 0:00:10
   -- ------------------------------------- 0.8/12.1 MB 1.3 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/12.1 MB 1.3 MB/s eta 0:00:09
   ---- ----------------------------------- 1.3/12.1 MB 1.3 MB/s eta 0:00:09
   ----- ---------------------------------- 1.6/12.1 MB 1.3 MB/s eta 0:00:09
   ------ --------------------------------- 1.8/12.1 MB 1.3 MB/s eta 0:00:09
   ------ --------------------------------- 2.1/12.1 MB 1.3 MB/s eta 0:00:08
   ------- -------------------------------- 2.4/12.1 MB 1.3 MB/s eta 0:00:08
   -------- ------------------------------- 2.6/12.1 MB 1.3 MB/s eta 0:00:08
   --------- ------------------------------ 2.9/12.1 MB 1.3 MB/s eta 0:00:08
   ---------- ------

In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# ── PART A: Real scrape attempt ──────────────────────────────────────────
headers = {"User-Agent": "Mozilla/5.0"}
url = "https://www.flightaware.com/live/airport/KLAX"

try:
    r = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")
    rows = soup.find_all("tr", class_=lambda x: x and "flight" in x.lower())
    print(f"Scraped {len(rows)} rows from FlightAware")
except Exception as e:
    print(f"Scrape blocked/failed (expected): {e}")

# ── PART B: Realistic mock flight dataset ────────────────────────────────
np.random.seed(42)
random.seed(42)

routes = [
    ("KLAX", "KLAS"), ("KLAX", "KSFO"), ("KLAX", "KASE"),
    ("KTEB", "KPBI"), ("KTEB", "KMIA"), ("KPDK", "KBHM"),
    ("KBOS", "KJFK"), ("KORD", "KDEN"), ("KDAL", "KHOU"),
]

aircraft_types = ["Citation XLS", "Gulfstream G450", "Phenom 300",
                  "King Air 350", "Challenger 350", "Learjet 75"]

# ✅ Define weights ONCE here, outside the function
weights = np.array([1,2,3,4,5,5,5,4,3,3,4,5,5,4,3,2], dtype=float)
hour_probs = weights / weights.sum()

def generate_flights(n_days=90):
    records = []
    base_date = datetime.now() - timedelta(days=n_days)

    for day_offset in range(n_days):
        current_date = base_date + timedelta(days=day_offset)
        weekday = current_date.weekday()

        base_demand = 8
        if weekday in [0, 3, 4]:
            base_demand = 15
        elif weekday in [5, 6]:
            base_demand = 6

        month = current_date.month
        if month in [6, 7, 8, 12]:
            base_demand = int(base_demand * 1.3)

        n_flights = np.random.poisson(base_demand)

        for _ in range(n_flights):
            origin, dest = random.choice(routes)
            # ✅ Clean usage — weights defined above, just referenced here
            hour = np.random.choice(range(6, 22), p=hour_probs)

            records.append({
                "date": current_date.date(),
                "datetime": current_date.replace(hour=hour, minute=random.randint(0, 59)),
                "flight_number": f"PVT{random.randint(1000,9999)}",
                "origin": origin,
                "destination": dest,
                "aircraft_type": random.choice(aircraft_types),
                "hour": hour,
                "weekday": weekday,
                "month": month,
            })
    return pd.DataFrame(records)

df_flights = generate_flights(n_days=90)
print(f"\n✅ Generated {len(df_flights)} flights over 90 days")
print(df_flights.head(10))

df_flights.to_csv("flights_raw.csv", index=False)
print("\n💾 Saved to flights_raw.csv")

Scraped 0 rows from FlightAware

✅ Generated 975 flights over 90 days
         date                   datetime flight_number origin destination  \
0  2026-01-26 2026-01-26 09:01:15.667087       PVT5506   KLAX        KSFO   
1  2026-01-26 2026-01-26 09:08:15.667087       PVT2679   KTEB        KPBI   
2  2026-01-26 2026-01-26 08:05:15.667087       PVT7912   KDAL        KHOU   
3  2026-01-26 2026-01-26 19:05:15.667087       PVT4582   KLAX        KLAS   
4  2026-01-26 2026-01-26 15:38:15.667087       PVT1434   KDAL        KHOU   
5  2026-01-26 2026-01-26 17:45:15.667087       PVT9928   KTEB        KPBI   
6  2026-01-26 2026-01-26 07:28:15.667087       PVT5557   KTEB        KPBI   
7  2026-01-26 2026-01-26 21:44:15.667087       PVT7924   KLAX        KASE   
8  2026-01-26 2026-01-26 18:09:15.667087       PVT4527   KTEB        KMIA   
9  2026-01-26 2026-01-26 10:05:15.667087       PVT7224   KLAX        KSFO   

     aircraft_type  hour  weekday  month  
0  Gulfstream G450     9        0      